___



# Excercise "Datenanalyse für Ingenieure" SS2026 - Data analysis & Forecasting</span>

___



---
# **1. Introduction**
---
Welcome to our exercise notebook on data analysis and time series forecasting with sktime! In this 2026 edition we additionally explore **foundation models** for time series — in particular **Chronos 2** by Amazon Science. This notebook will explore some tools for analyzing data and further show a small pipline approach to forecast future values. This notebook will provide you with some insights and hands-on experience in working with time series data. Therefore, let's get started and dive into the exciting world of data analysis and time series forecasting!

Please note it is mandatory to install all the required software and packages using the guide provided in Ilias before proceeding with this exercise notebook. The guide contains important instructions how to properly set up your environment. This ensures that all the necessary dependencies are installed. Failure to follow the instructions may result in errors or unexpected behavior while working through the notebook.

This is an interactive notebook and also includes some work assignments. Typical tasks are adding lines of code, documenting observations. Work orders are always marked in the color <span style="color:#A00000"> **red** </span>. We suggest you work in pairs or small groups so that you can share observations and discuss the tasks together.


___

## Agenda

1. Introduction, Agenda, learning goals and data loading

2. Data Analysis

3. Forecasting Excercise

4. Probalistic Forecasting

___

## Learning Goals (Sorted by Chapter)

### 📈 Data Analysis and Time Series Analysis

- Identify key properties of time series, such as **seasonality**, **trend**, and **autocorrelation**  
- Analyze energy-related time series and **detect**, **interpret**, and **evaluate** seasonal patterns  
- **Create and select** relevant calendar-based features using insights from seasonality and autocorrelation analysis

---

### 🤖 Machine Learning

- Understand the **core concepts** of the `sktime` library  
- Be familiar with **standard evaluation metrics** used in forecasting  
- **Evaluate forecasters**, interpret their errors, and **derive features** to improve model performance  
- Understand the **foundations of probabilistic forecasting** and its practical applications  
- Apply **zero-shot foundation models** (e.g. **Chronos 2**) for time series forecasting

___



## ⚡ IEEE Case 9 — Data & AI in Energy Systems

![IEEE Case 9](case_9_excercise_dataanalysis.png)

The **IEEE 9-Bus Test Case** is a simplified power system model ideal for exploring how **AI** and **data analysis** can improve modern energy grid operations.

---

### 🔍 Why AI Matters in Energy

As energy systems grow more complex, **AI enables smarter, faster, and more adaptive solutions** — from predicting failures to optimizing power flow.

---

### 🤖 Core AI Applications in Power Systems

| Area                   | Description                                                                 |
|------------------------|-----------------------------------------------------------------------------|
| **Forecasting**        | Predict load, generation (e.g., wind/solar), or market prices to support planning and operations. |
| **Predictive Maintenance** | Detect early signs of equipment failure using historical and real-time sensor data. |
| **Optimal Power Flow (OPF)** | Use AI to find fast, near-optimal solutions for generation dispatch and voltage control. |
| **Scheduling & Dispatch**   | Apply AI (e.g. reinforcement learning) to automate generation scheduling under uncertainty. |
| **Anomaly Detection**  | Spot faults or abnormal patterns in grid behavior using unsupervised learning. |

---

In [ ]:
#
# It is a try, but not a complete solution. A lot of messages are still in the notebook.
#
import pandas as pd
import logging
import warnings
import torch

logging.getLogger("pytorch_lightning").setLevel(logging.CRITICAL)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

#
# Seeded for reproducibility that everyone gets the same results.
#

torch.manual_seed(0)

In [ ]:
# The data is from the Open Power System Data project, which provides household electricity consumption data.
data = pd.read_csv('https://data.open-power-system-data.org/household_data/2020-04-15/household_data_60min_singleindex.csv', date_format='%Y-%m-%dT%H:%M:%SZ', index_col = "utc_timestamp", parse_dates=True , sep=',')
data.index = pd.to_datetime(data.index, utc=True)


## Data loading and first preprocessing
As a basis for the data analysis we need data in the first place. This publically available data set is described here: [Full Dataset Introduction](https://data.open-power-system-data.org/household_data/2020-04-15)

Here we have taken only a subset of the data, since we want to deal with only one building. Our choice is the industrial building 3. All the data is scaled in kWh. The building has an installed pv and an energy demand.



In [ ]:
import statsmodels.api as sm
from matplotlib import pylab
from pylab import *


# Basic configuration to get beautiful pictures
pylab.rcParams['figure.figsize'] = (16, 9)

# Get the relevant data for this excercise and resample it to hourly resolution to save runtime complexity
data["demand"] = data["DE_KN_industrial3_grid_import"].diff(1)
data["solar"] = (data["DE_KN_industrial3_pv_facade"].diff(1) + data["DE_KN_industrial3_pv_roof"].diff(1))

# shift the index of the data by one interval to get the correct time alignment
data.index = data.index - pd.Timedelta(hours=1)

# Omit data without values
data = data[["demand","solar"]].dropna()
# Let the data start with a full day and end with a full day
data =  data[(data.index >= pd.to_datetime("2016-03-01",utc=True)) &  (data.index < pd.to_datetime("2017-05-01",utc=True))]
data = data.asfreq('1h')

---
# **2. Data Analysis**
---
<img src="https://imgs.xkcd.com/comics/data_trap.png" width="400" height="400">

[This xkcd comic you can find here](https://xkcd.com/2582/)

Within the data analysis chapter, we will first use simple tools from Pandas to get an overview of the data set.
After that, we will make a daily observation of the load and analyse the difference between a weekday and weekend.
In the last part we will use more complex tools like autocorellation plots and a seasonal decomposition to identify properties like trend and seasonality in our time series.

First of all, we would like to show you three very simple functions that Pandas has ready for you :

## Warmup and get familiar with the data

1. head() First five rows of the data set. Commonly used as a sanity check to see how the Data is constructed. [API](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html)
2. describe() Provides basic satistical values of the dataset. For example, mean, standard deviation and quantiles.[API](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)
3. plot() The plot function draws a simple plot over all collumns of the given dataset with mathplolib. [API](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html)

### <span style="color:#A00000"> Use the three functions (head, describe and plot) below! </span>



In [ ]:
# Use the head function
# TODO 

In [ ]:
# Use the describe function
# TODO 

 ### <span style="color:#A00000 "> Discuss in your group: </span>
 - <span style="color:#A00000 "> Based on these broad statistic, what would you infer about the house considered? </span>
 - <span style="color:#A00000 "> Do these statistics help you to get a feel for the data and what it looks like? </span>
 - <span style="color:#A00000 "> Does this data seem realistic? </span>
 - <span style="color:#A00000 "> Do you see any challanges? </span>
 - <span style="color:#A00000 "> How much data do we have? </span>

< Space for your answers>


In [ ]:
# Use the plot function
# TODO 

 ### <span style="color:#A00000 "> Point out two observations about the given data! </span>

1. < Observation 1 >
2. < Observation 2 >

 ### <span style="color:#A00000 "> Discuss in your group: </span>
 - <span style="color:#A00000 "> What did you find more beneficial, the statistics of the plots? </span>
 - <span style="color:#A00000 "> Do you see a benefit in both statstics and plots or would you only consider one of them? </span>

< Space for your answers>


## Daily observations

This part first shows an example of the pivot table how it is used to plot the data on daily basis. Further it extends by using the month attribute to plot the months in different plots. As you can see differnt days of the month are plotted.

In [ ]:
# Calculating the hour of the day the weekday and the day since the beginning of the time series to create the pivot table
data["hour"] = data.index.hour.values
data["weekday"] = data.index.weekday.values
data["month"] = data.index.month.values
data["days_since_start"] = [int(x/(24)) for x in range(0,len(data))]

# creates the pivot table to get a table with the days since start in the columns and hours of the day as rows. For later usage the months are taken into account as the value parameter.
pivot_solar = pd.pivot_table(data, index=['hour'],columns=['days_since_start'], values=['solar','month'])
pivot_solar["solar"]


In [ ]:
fig, ax = plt.subplots(1,len(data["month"].unique()))
for i in range(len(data["month"].unique())):
    pivot_solar[pivot_solar["month"]==data["month"].unique()[i]]["solar"].plot(ax=ax[i],figsize=(30, 4), layout= (7,1),ylim = (0,20),legend=False, colormap="summer", title="Month : " + str(data["month"].unique()[i]))

 ### <span style="color:#A00000 "> Discuss in your group: </span>
 - <span style="color:#A00000 "> What do you observe during the different months ? </span>

### Workingdays

 ### <span style="color:#A00000 "> Plot only the working days (i.e. Monday-Friday):</span>
 - <span style="color:#A00000 ">The aim is to first filter the data to only get the days from Monday to Friday.</span>
 - <span style="color:#A00000 ">Create a variable ``working_day_data`` and use the weekday column in ``data`` to select the working days. The weekday column is enumerated from 0-6 with 0 being monday and 6 sunday. Therefore working days are the days with a weekday value smaller than five. You can select a column in pandas with ``data["column_name"]`` and if you want to select a subset of the data based on the value in this column you need to use the syntax ``data[data["column_name"] * x]``, where ``*`` indicates a mathematical operater such as ``<`` and ``x`` is the condition. For example, to only select Tuesday you would use ``tuesday_data = data[data["weekday"] == 1]``.</span>
 - <span style="color:#A00000 ">Create the pivot table similar to the example above using the function pivot_table from pandas! [API](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html). Make sure you use the filtered ``working_day_data`` variable created above.</span>
 - <span style="color:#A00000 ">Extend the pivot table to plot working days. Use "days_since_start" as ``columms``, "hour" as ``index`` and "demand" as ``values``!</span>

In [ ]:

# this get's gapped
working_day_data = # TODO 
pivot_workingdays = # TODO 

# plots the data
pivot_workingdays["demand"].plot(legend=False,colormap="summer")



### Weekend

 ### <span style="color:#A00000 "> Plot the weekends:</span>
 - <span style="color:#A00000 ">Repeat the task above, but this time only select the weekends (remember weekday is enumerated from 0-6, with 0 being Monday).</span>

In [ ]:

# this get's gapped
weekday_data = # TODO 
pivot_weekends = # TODO 
pivot_weekends["demand"].plot(legend=False,colormap="summer")


In [ ]:
#
# Plot the median of all weekdays
#

pivot_workingdays.median(axis=1).plot(color="red", label="Working Median")
pivot_workingdays.quantile(0.25,axis=1).plot(color="red",linestyle='dotted', label="Working 0.25 Quantile")
pivot_workingdays.quantile(0.75,axis=1).plot(color="red",linestyle='dotted', label="Working 0.75 Quantile")
pivot_weekends.median(axis=1).plot(color="blue", label="Weekend Median")
pivot_weekends.quantile(0.25,axis=1).plot(color="blue",linestyle='dotted', label="Weekend 0.25 Quantile")
pivot_weekends.quantile(0.75,axis=1).plot(color="blue",linestyle='dotted', label="Weekend 0.75 Quantile")
plt.legend()

 ### <span style="color:#A00000 "> Discuss in your group:</span>
 - <span style="color:#A00000 "> Do these different plots (weekdays, weekends, median) fit your expectations?</span>
 - <span style="color:#A00000 "> What could explain the pattern for this building for a weekday? </span>
 - <span style="color:#A00000 "> Could you interfer the base load of the building by looking at the daily observation plots? </span>
 - <span style="color:#A00000 "> How could explain the higher variance of weekdays in contrast to weekends? </span>
 - <span style="color:#A00000 "> Based on these observations, what features would you consider extracting or engineering for a forecasting task? </span>

< Space for your answers>


## Autorcorrelation Function and Seasonal Decomposition

### Autocorrelation Function Plot


The autocorrelation function (ACF) is a statistical technique that we can use to identify how correlated the values in a time series are with each other. The ACF plots the correlation coefficient against the lag, which is measured in terms of a number of periods or units. [Explanation from here](https://www.baeldung.com/cs/acf-pacf-plots-arma-modeling#:~:text=The%20autocorrelation%20function%20(ACF)%20is,number%20of%20periods%20or%20units.)

An detailed expleanation is here: [Autocorrelation](https://support.minitab.com/en-us/minitab/21/help-and-how-to/statistical-modeling/time-series/how-to/autocorrelation/interpret-the-results/autocorrelation-function-acf/)

API-Statsmodels [API](https://www.statsmodels.org/devel/generated/statsmodels.tsa.stattools.acf.html)

### Seasonal Decomposition

An detailed explaination about the used seasonal decomposition can be found here: [API](https://www.statsmodels.org/dev/generated/statsmodels.tsa.seasonal.seasonal_decompose.html)

The additive model which is used here is defined as $Y[t] = T[t] + S[t] + e[t]$.

The results are obtained by first estimating the trend by applying a convolution filter to the data. The trend is then removed from the series and the average of this de-trended series for each period is the returned seasonal component.

#### Demand ACF and Seasonal Decomposition

 ### <span style="color:#A00000 "> Play around with autocorrelation: </span>
- <span style="color:#A00000 "> Use the ``sm.tsa.graphics.plot_acf(x, lags=None)`` function to plot the autocorrelation function of ``data["demand"]``. </span>
- <span style="color:#A00000 "> Play around with a different number of lags, i.e. 5, 24, 200. </span>
- <span style="color:#A00000 "> What do you observe? </span>

In [ ]:
# TODO 

 ### <span style="color:#A00000 "> Play around with seasonal decomposition: </span>
- <span style="color:#A00000 "> Use the ``sm.tsa.seasonal_decompose(x, period=None)`` function to create a seasonal decomposition of ``data["demand"]``. </span>
- <span style="color:#A00000 "> Visualise this decomposition with ``sm.tsa.seasonal_decompose().plot()``. </span>
- <span style="color:#A00000 "> Adjust the ''period'' parameter of the function, try for example 24 (a day), or 168 (a week), or something random (e.g. 77). </span>
- <span style="color:#A00000 "> What do you observe? </span>

In [ ]:
# do seasonal decomposition of demand here

# TODO 


### ACF and Seasonal Decomposition Solar

 ### <span style="color:#A00000 "> Repeat the above two tasks for the solar data! </span>

In [ ]:
# do seasonale decomposition on solar data (data["solar"])
# Gap here for acf
# TODO 

In [ ]:
# do seasonal decomposition of demand here
# this get's gapped
# TODO 


# <span style="color:#A00000 ">  STOP HERE - We want to discuss some things together before going on </span>

---
# **3. Forecasting Exercise**
---
In this section of the Jupyter Notebook, you will learn how to use **[sktime](https://www.sktime.net/en/stable/)** for time series forecasting.

Time series forecasting is a technique used to predict future values of a variable based on historical data — and optionally known exogenous variables.  
It is widely used in areas such as:

- Finance  
- Economics  
- Engineering  
- **Energy Informatics**

---

## 🔗 Useful Resources

- 📘 **Basic sktime API description:**  
  [sktime API](https://www.sktime.net/en/stable/)

- 🔮 **List of available forecasters:**  
  [Forecaster Overview](https://www.sktime.net/en/stable/estimator_overview.html#filter=all&tags=%7B%7D)

- ⏳ **Forecasting Horizon documentation:**  
  [ForecastingHorizon](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.forecasting.base.ForecastingHorizon.html)

- 🧩 **ExpandingWindowSplitter:**  
  [ExpandingWindowSplitter](https://www.sktime.net/en/v0.21.0/api_reference/auto_generated/sktime.forecasting.model_selection.ExpandingWindowSplitter.html)

---

## 📊 Time Series Specific - Expanding Window Splitter Illustration

    Fold 1:  *  *  *  *  *  x  x  x  -  -  -
    Fold 2:  *  *  *  *  *  *  x  x  x  -  -
    Fold 3:  *  *  *  *  *  *  *  x  x  x  -
    Fold 4:  *  *  *  *  *  *  *  *  x  x  x
             |  |  |  |  |  |  |  |  |  |  |
    Pos:     1  2  3  4  5  6  7  8  9 10 11

**Legend:**

- `*` → Training window  
- `x` → Forecasting horizon (test window)  
- `-` → Future/unseen data  
- Each row is a fold in cross-validation using an expanding window.
- The training section expands while the test window slides forward.

## 🧪 Exercises

The following exercises will guide you through creating a simple time series forecast using **sktime**.

You will apply a **basic sktime forecasting workflow** to data already introduced in the previous exercise.


## 🔍 Let's Look at What We Prepared

### 📊 Data

We use the data from the previous exercise part.

---

### 📏 Metrics

To evaluate the accuracy of the forecast, this section introduces various metrics such as:

- **Mean Absolute Percentage Error (MAPE)**
- **Mean Absolute Error (MAE)**
- **Mean Squared Error (MSE)**

These metrics help you understand how close the forecasted values are to the actual values.

Mathematical definitions:

- $ \text{MAPE} = \frac{1}{n} \sum_{t=1}^{n} \left| \frac{A_t - F_t}{A_t} \right| $
- $ \text{MAE} = \frac{1}{n} \sum_{t=1}^{n} \left| A_t - F_t \right| $
- $ \text{MSE} = \frac{1}{n} \sum_{t=1}^{n} \left( A_t - F_t \right)^2 $

 # <span style="color:#A00000 "> Discuss in Groups</span>
 - <span style="color:#A00000 ">What are the differnt aims of the Metrics/ Loss Functions?</span>





---

### 🤖 Models

We compare three forecasters of increasing complexity:

1. **NaiveForecaster** — classical baseline (e.g. *yesterday's value*).
2. **N-HiTS** — *[Neural Hierarchical Interpolation for Time Series Forecasting](https://arxiv.org/abs/2201.12886)* (Challu et al., 2022): a deep-learning model that has to be **trained** on our data.
3. **Chronos 2** — *[Chronos-2: From univariate to universal forecasting](https://www.amazon.science/blog/introducing-chronos-2-from-univariate-to-universal-forecasting)* (Amazon Science, 2025): a **pre-trained foundation model** for time series that works **zero-shot**, supports **exogenous covariates**, and natively produces **probabilistic forecasts**.


---

### 📈 Visualization

To help interpret the results, we include visualizations that compare **forecasted values** to **actual values** over time.

These visualizations help identify:

- Trends
- Seasonal patterns
- Anomalies or mismatches between forecast and ground truth

---

### ✅ Summary

This section provides a **practical introduction** to using `sktime` for time series forecasting.  
By the end, you will have understood how to:

- Apply a basic forecasting model
- Evaluate its performance using multiple metrics
- Visualize the results for better interpretation


In [ ]:
from sklearn.preprocessing import StandardScaler

from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.pytorchforecasting import PytorchForecastingNHiTS
from sktime.forecasting.chronos2 import Chronos2Forecaster

from sktime.utils.plotting import plot_series , plot_windows
from sktime.forecasting.model_evaluation import evaluate
from sktime.split import ExpandingWindowSplitter

from sktime.performance_metrics.forecasting import MeanAbsoluteError , MeanSquaredError , MeanAbsolutePercentageError



#
# Dummy Demonstration How the basic sktime workflow for prediction works within a naive forecaster 
#


# step 1: data specification
y = data["demand"][:24*10] 
# step 2: specifying forecasting horizon
fh = np.arange(1, 25)
# step 3: specifying the forecasting algorithm
forecaster = NaiveForecaster(strategy="last", sp=24*7)
# step 4: fitting the forecaster
forecaster.fit(y)
# step 5: querying predictions
y_pred = forecaster.predict(fh)
# optional: plotting predictions and past data
plot_series(y, y_pred, labels=["y", "y_pred"])


# <span style="color:#A00000 "> Discuss with your group: </span>
- <span style="color:#A00000 "> Discuss what the naiv forecaster does? </span>
- <span style="color:#A00000 "> Why could a navie forecaster could be useful?</span>
- <span style="color:#A00000 "> What is the basic assumption behind a naive forecaster?</span>
- <span style="color:#A00000 "> Where a naive forecaster fails and why?</span>
- <span style="color:#A00000 "> How can other forecaster prevent this?</span>

In [ ]:
#
#
# HELPER METHODS BLOCK Do not change this block
#
#



from ipywidgets import interact, IntSlider, fixed


def plot_forecasts_for_day(models_dict, day):
    """
    Plot forecasts for a given day, using the native timestamp index from y_test.

    Args:
        models_dict (dict): Keys are model names, values are DataFrames with 'y_pred' and 'y_test'.
        day (int): Day index (1-based).
    """
    first_df = next(iter(models_dict.values()))
    y_true = first_df.iloc[day - 1]["y_test"]

    x = y_true.index
    y_true = np.array(y_true)

    plt.figure(figsize=(12, 5))
    plt.plot(x, y_true, label="y (True)", linewidth=2)

    for label, df in models_dict.items():
        y_pred = df.iloc[day - 1]["y_pred"]
        if hasattr(y_pred, "index"):
            y_pred = np.array(y_pred)
        plt.plot(x, y_pred, label=label, linestyle="--")

    plt.title(f"Forecast Comparison – {x[0].date()}")
    plt.xlabel("Timestamp")
    plt.ylabel("Value")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

def interactive_forecast_plot(models_dict):
    """
    Interactive slider to browse forecast plots per day using existing timestamped y_test.
    """
    num_days = len(next(iter(models_dict.values())))

    interact(
        plot_forecasts_for_day,
        models_dict=fixed(models_dict),
        day=IntSlider(min=1, max=num_days, step=1, value=1, description="Day")
    )
#
#
#

def plot_full_forecast_series(models_dict):
    """
    Plot the full time series of y_test and all y_pred across all days, using native timestamps.

    Assumes y_test and y_pred in each row are pandas Series (with DateTimeIndex).
    """
    # Get full y_true time series
    first_df = next(iter(models_dict.values()))
    y_true_series = pd.concat(first_df["y_test"].values)

    plt.figure(figsize=(14, 5))
    plt.plot(y_true_series.index, y_true_series.values, label="y (True)", linewidth=2)

    # Plot all model predictions
    for label, df in models_dict.items():
        y_pred_series = pd.concat(df["y_pred"].values)
        plt.plot(y_pred_series.index, y_pred_series.values, label=label, linestyle="--")

    plt.title("Full Forecast Series Over Time")
    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.xticks(rotation=45)
    plt.show()


def plot_full_forecast_series_prob(models_dict):
    """
    Plot the full time series of y_test and all y_pred across all days, using native timestamps.

    Assumes y_test and y_pred in each row are pandas Series (with DateTimeIndex).
    """
    # Get full y_true time series
    first_df = next(iter(models_dict.values()))
    y_true_series = pd.concat(first_df["y_test"].values)

    plt.figure(figsize=(14, 5))
    plt.plot(y_true_series.index, y_true_series.values, label="y (True)", linewidth=2)

    # Plot all model predictions
    for label, df in models_dict.items():
        y_pred_df = pd.concat(df["y_pred_quantiles"].values)
        # remove the multiindex from the DataFrame
        y_pred_df.columns = y_pred_df.columns.droplevel(0)  #
        #iterate over the quantiles
        for quantile in y_pred_df.columns:
            y_pred_series = y_pred_df[quantile]
            plt.plot(y_pred_series.index, y_pred_series.values, label=f"{label} - {quantile}", linestyle="--")
        

    plt.title("Full Forecast Series Over Time")
    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.xticks(rotation=45)
    plt.show()



#
# Summary Metrics Function
#

def summarize_metrics(models_dict, prefix="test_"):
    """
    Generate an evaluation table from precomputed test metrics in the DataFrames.

    Args:
        models_dict (dict): Dictionary with model names as keys and DataFrames as values.
                            Each DataFrame should contain columns like 'test_MAE', 'test_MSE', etc.
        prefix (str): Prefix of metric columns (default is 'test_').

    Returns:
        pd.DataFrame: Evaluation table with models as rows and metrics as columns (mean per metric).
    """
    results = {}

    for label, df in models_dict.items():
        # Select only columns with the given prefix
        metric_cols = [col for col in df.columns if col.startswith(prefix)]
        
        # Remove the prefix in column names
        renamed_metrics = {col: col[len(prefix):] for col in metric_cols}
        
        # Compute mean for each metric
        means = df[metric_cols].mean().rename(index=renamed_metrics)
        
        results[label] = means

    # sort by the first metric
    df_res = pd.DataFrame.from_dict(results, orient="index").round(4)
    df_res = df_res.sort_values(by=df_res.columns[0], ascending=True)

    return df_res

#
# Helper Methods End
#

# <span style="color:#A00000 "> Discuss with your group: </span>
- <span style="color:#A00000 "> Discuss what the naiv forecaster does? </span>
- <span style="color:#A00000 "> Why could a navie forecaster could be useful?</span>
- <span style="color:#A00000 "> What is the basic assumption behind a naive forecaster?</span>
- <span style="color:#A00000 "> Where a naive forecaster fails and why?</span>
- <span style="color:#A00000 "> How can other forecaster prevent this?</span>

In [ ]:
y = data["demand"]

TRAINING_LENGTH = 366 * 24 # Because 2016 was a leap year, we use 366 days of hourly data

cv = ExpandingWindowSplitter(
    step_length=24, fh=[i for i in range(1,25)], initial_window=TRAINING_LENGTH
)

plot_windows(cv=cv, y=y)

metrics = [MeanSquaredError(), MeanAbsoluteError(),MeanAbsolutePercentageError()]
# build up a dictionary to store the results
results_dict = {}

<span style="color:#A00000">

### GET YOUR OWN NHiTS 🤖

- Use the `StandardScaler` as a Pipeline in combination with NHiTS using the `*` operator  
- Use the class `PytorchForecastingNHiTS()` and instantiate it with the `MODEL_PARAMS`, the `TRAINER_PARAMS`, and the `DATASET_PARAMS`

</span>

In [ ]:
#
# Forecaster Evaluation
#

#
#   NaiveForecaster
#

forecaster_naive = NaiveForecaster(strategy="last", sp= 24 * 7)
df_naiv_168 = evaluate(forecaster=forecaster_naive, y=y, cv=cv, return_data=True,scoring=metrics)
results_dict["NaiveForecaster"] = df_naiv_168


#
#   NHits Parameters for PytorchForecastingNHiTS
#

MODEL_PARAMS={
        "context_length": 24 * 7,
        "prediction_length": 24,
        "hidden_size": 64,
        }

TRAINER_PARAMS={
        "max_epochs": 5,
    }

DATASET_PARAMS={
        "max_encoder_length": 24 * 7,  # Context length
        "max_prediction_length": 24,    # Prediction length
    }


#
#
#
  
forecaster_nhits = # TODO 


df_nhits = evaluate(forecaster=forecaster_nhits, y=y, cv=cv,return_data=True, return_model=True, strategy="no-update_params",scoring=metrics)
results_dict["NHits"] = df_nhits



<span style="color:#A00000">

### GET YOUR OWN CHRONOS 2 🤖

[Chronos 2](https://huggingface.co/amazon/chronos-2) is a **pre-trained foundation model** for time series forecasting from Amazon Science. Unlike N-HiTS, it does **not need to be trained** on our data — it forecasts **zero-shot**.

- Instantiate `Chronos2Forecaster` with the model `"autogluon/chronos-2-small"` (28M parameters — small enough to run on CPU / free Colab GPU).
- Pass a `config` dictionary with at least `device_map` (`"cuda"` if a GPU is available, otherwise `"cpu"`).
- Reuse the same `evaluate(...)` call as before — Chronos 2 plugs into sktime's standard forecaster interface.

</span>

In [ ]:
#
#   Chronos 2 — Zero-Shot Foundation Model (no exogenous features)
#

import torch

CHRONOS_CONFIG = {
    "device_map": "cuda" if torch.cuda.is_available() else "cpu",
    "context_length": 24 * 7,
}

forecaster_chronos2 = # TODO  instantiate Chronos2Forecaster("autogluon/chronos-2-small", config=CHRONOS_CONFIG)

df_chronos2 = evaluate(
    forecaster=forecaster_chronos2,
    y=y,
    cv=cv,
    return_data=True,
    return_model=False,
    strategy="no-update_params",
    scoring=metrics,
)
results_dict["Chronos 2"] = df_chronos2


## First Evaluation

<span style="color:#A00000">

### Discuss in your group 🤖

- Execute the next 3 cells!
- How well did the model perform?  
- Were the results as expected?

</span>

In [ ]:
plot_full_forecast_series(results_dict)

In [ ]:
#
# Outcommented for performance reasons use it if you want to see the interactive plot of each day
# 

# interactive_forecast_plot(result_dict)

In [ ]:
summarize_metrics(results_dict, prefix="test_")

## Enrich the forecast with features

Information about the features can be found [here](https://www.sktime.net/en/latest/api_reference/auto_generated/sktime.transformations.series.date.DateTimeFeatures.html).

<span style="color:#A00000">

# Add day and hour features + Holiday features

- Use `DateTimeFeatures()` with the time series frequency **hourly**  
- Manually select two features: `"day_of_week"`, `"hour_of_day"`

</span>



In [ ]:

from sktime.transformations.series.date import DateTimeFeatures
from sktime.transformations.series.holiday import HolidayFeatures
from holidays import country_holidays

holiday_transformer = HolidayFeatures(
   calendar=country_holidays(country="DE",state="BW"),
   include_bridge_days=True,
   )  
X_holiday = holiday_transformer.fit_transform(y)
X_holiday["holiday"] = X_holiday.sum(axis=1)


calendar_transformer = DateTimeFeatures(ts_freq=# TODO , manual_selection= # TODO )
X_calendar = calendar_transformer.fit_transform(y)

#join them together as a feature matrix
X = pd.concat([X_holiday["holiday"], X_calendar], axis=1)


<span style="color:#A00000">

# Provide Additional Features to Your Model

Evaluate your N-HiTS model using additional features.  
Be sure to set `return_model = True` when calling the `evaluate()` function — we want to reuse the trained model.  
Also, pass the exogenous features using the `X` parameter with your prepared feature data.

</span>




In [ ]:
#
#   NHiTS with Features
#  

forecaster_nhits_features = # TODO 
)

df_nhits_features = evaluate(forecaster=forecaster_nhits_features,y=y,X=X,cv=cv,return_data=True, strategy="no-update_params",scoring=metrics, return_model=True)
results_dict["NHits with Calendaric Features"] = df_nhits_features


<span style="color:#A00000">

### Chronos 2 with Calendar & Holiday Features 🤖

A major new capability of **Chronos 2** is **native support for exogenous covariates**:

- The values of `X` that fall in the training window become **past covariates**.
- The values of `X` that fall in the forecast horizon become **future covariates** (we already know calendar / holiday information for the future!).

Evaluate `Chronos2Forecaster` again, this time passing the feature matrix `X` to `evaluate(..., X=X, ...)`.

</span>

In [ ]:
#
#   Chronos 2 with Calendar & Holiday Features
#

forecaster_chronos2_features = # TODO  instantiate Chronos2Forecaster with model "autogluon/chronos-2-small" and CHRONOS_CONFIG

df_chronos2_features = evaluate(
    forecaster=forecaster_chronos2_features,
    y=y,
    X=X,
    cv=cv,
    return_data=True,
    return_model=False,
    strategy="no-update_params",
    scoring=metrics,
)
results_dict["Chronos 2 with Calendaric Features"] = df_chronos2_features


## Final Deterministic Evaluation

<span style="color:#A00000">

### Discuss in your group 🤖

- Execute the next 3 cells!
- How well did the model perform?  
- Were the results as expected?

</span>

In [ ]:
plot_full_forecast_series(results_dict)

In [ ]:
#
# Outcommented for performance reasons use it if you want to see the interactive plot of each day
# 

# interactive_forecast_plot(result_dict)

In [ ]:
summarize_metrics(results_dict, prefix="test_")

---

# **4. Probalistic Forecasting**
---

It is often desirable to not only produce a point forecast, but also to estimate the uncertainty of the forecast.

### 📊 What is a Probabilistic Forecast?

A **probabilistic forecast** provides a **range of possible future values**, not just a single prediction.

---

#### 🔹 Point Forecast (Standard):
Predicts only one value per time step.  
*Example:*  
> Tomorrow's sales will be **100 units**.

---

#### 🔹 Probabilistic Forecast:
Gives multiple quantiles or intervals — showing the **uncertainty** around the prediction.  
*Example:*  
> There's a 90% chance that tomorrow's sales will be **between 85 and 120 units**.

---

### ✅ Why Use It?

- Captures **uncertainty** in your forecasts
- Supports **risk-aware decision making**
- Essential for **planning in unpredictable environments**

---



<span style="color:#A00000">

# Instruction: Create a Probabilistic Forecast

Use the `evaluate` function to generate a probabilistic forecast with the trained model. Make sure to set the scoring to a probalistic metric e.g. PinballLoss  
Use the exogenous features `X` as input.  

</span>



In [ ]:
from sktime.performance_metrics.forecasting.probabilistic import PinballLoss
# Quantiles
quantiles = [0.1, 0.5, 0.9]  # 10%, 50%, and 90% quantiles

loss = PinballLoss(alpha=quantiles)
result_dict_prob = {}

#
# Chronos 2 natively produces quantile forecasts — no special configuration needed.
# Simply pass the same forecaster instance and use `PinballLoss` as the scoring function.
#
forecaster_chronos2_prob = # TODO  instantiate Chronos2Forecaster("autogluon/chronos-2-small", config=CHRONOS_CONFIG)

df_chronos2_prob = evaluate(
    forecaster=forecaster_chronos2_prob,
    X=X,
    y=y,
    cv=cv,
    return_data=True,
    return_model=False,
    strategy="no-update_params",
    scoring=loss,
)
result_dict_prob["Chronos 2"] = df_chronos2_prob


## Probalistic Evaluation

<span style="color:#A00000">

### Discuss in your group 🤖

- Execute the next cell!
- How well did the model perform?  
- Were the results as expected?

</span>

In [ ]:
plot_full_forecast_series_prob(result_dict_prob)

---

# **Thank you for your Attention !**

--- 
## <span style="color:#A00000 "> Now you can play! </span>
- <span style="color:#A00000 "> Play around with the forecast horizon and the number of historical features.</span>
- <span style="color:#A00000 "> You can also try altering the scope variable being forecast.</span>
- <span style="color:#A00000 "> How do the results change?</span>
- <span style="color:#A00000 "> Which forecasters perform best?</span>

<img src="https://imgs.xkcd.com/comics/machine_learning.png" width="600" height="800">

[This xkcd comic you can find here](https://xkcd.com/1838/)
